# 🩺 Breast Cancer Classification
## Mid-Project — Comparing Decision Tree, Naive Bayes, KNN & ANN

**Dataset:** Wisconsin Breast Cancer Dataset (scikit-learn built-in)  
**Source / Link:** https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html  
**Original UCI Repository:** https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(diagnostic)  

**Problem Type:** Binary Classification (Malignant = 0, Benign = 1)  
**Goal:** Predict whether a tumour is malignant or benign based on 30 numeric features computed from digitised images of fine-needle aspirates.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

# Styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries imported successfully ✓')

## 2. Load & Explore Dataset

In [ ]:
# Load the Wisconsin Breast Cancer dataset
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print('Dataset shape:', df.shape)
print('\nFeature names:', list(data.feature_names))
print('\nTarget classes:', data.target_names)  # 0=malignant, 1=benign
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe().T.round(2)

In [ ]:
print('Class distribution:')
print(df['target'].value_counts())
print(f'  0 = Malignant: {(df.target==0).sum()} samples')
print(f'  1 = Benign:    {(df.target==1).sum()} samples')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution bar
df['target'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#C44E52','#4C72B0'], edgecolor='white')
axes[0].set_xticklabels(['Malignant (0)','Benign (1)'], rotation=0)
axes[0].set_title('Class Distribution'); axes[0].set_ylabel('Count')

# Feature correlation heatmap (first 10 features for readability)
corr = df.iloc[:, :10].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', ax=axes[1], linewidths=0.3)
axes[1].set_title('Feature Correlation (first 10 features)')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# ── 3.1 Check for Missing Values ──────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found ✓')

In [ ]:
# ── 3.2 Encoding ─────────────────────────────────────────────────────────
# All features are already numeric; the target (0/1) requires no encoding.
print('Feature dtypes:')
print(df.dtypes.value_counts())
print('\nNo categorical columns — encoding step not required ✓')

In [ ]:
# ── 3.3 Train / Test Split (80 / 20, stratified) ─────────────────────────
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')

In [ ]:
# ── 3.4 Feature Scaling (StandardScaler) ─────────────────────────────────
# Required for KNN and ANN; Decision Tree and Naive Bayes use raw data.
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit on train only
X_test_sc  = scaler.transform(X_test)

print('Feature scaling applied (StandardScaler) ✓')
print(f'Train mean after scaling (first feature): {X_train_sc[:,0].mean():.6f}')
print(f'Train std  after scaling (first feature): {X_train_sc[:,0].std():.6f}')

## 4. Train & Evaluate Models

In [ ]:
# Helper to evaluate and store results
results = {}

def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    cm  = confusion_matrix(y_te, y_pred)
    cr  = classification_report(y_te, y_pred,
              target_names=['Malignant','Benign'], output_dict=True)
    results[name] = {'model': model, 'acc': acc, 'cm': cm, 'cr': cr, 'pred': y_pred}
    print(f'\n{'='*48}')
    print(f'  {name}   →   Accuracy: {acc:.4f} ({acc*100:.2f}%)')
    print(f'{'='*48}')
    print(classification_report(y_te, y_pred, target_names=['Malignant','Benign']))
    return acc

In [ ]:
# ── 4.1 Decision Tree ─────────────────────────────────────────────────────
dt = DecisionTreeClassifier(random_state=42)
evaluate('Decision Tree', dt, X_train, X_test, y_train, y_test)

In [ ]:
# ── 4.2 Naive Bayes ───────────────────────────────────────────────────────
nb = GaussianNB()
evaluate('Naive Bayes', nb, X_train, X_test, y_train, y_test)

In [ ]:
# ── 4.3 K-Nearest Neighbours (k=5) ───────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=5)
evaluate('KNN', knn, X_train_sc, X_test_sc, y_train, y_test)

In [ ]:
# ── 4.4 Artificial Neural Network (MLP) ──────────────────────────────────
ann = MLPClassifier(
    hidden_layer_sizes=(64, 32),  # two hidden layers
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)
evaluate('ANN', ann, X_train_sc, X_test_sc, y_train, y_test)

## 5. Model Comparison & Visualisation

In [ ]:
# ── 5.1 Accuracy Comparison ───────────────────────────────────────────────
names  = list(results.keys())
accs   = [results[n]['acc'] for n in names]
colors = ['#4C72B0','#DD8452','#55A868','#C44E52']

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(names, accs, color=colors, width=0.5,
              edgecolor='white', linewidth=1.2)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.004,
            f'{acc:.4f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')
ax.set_ylim(0.85, 1.0)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy Comparison – Breast Cancer Dataset',
             fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
ax.axhline(y=max(accs), color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2 Confusion Matrices ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.flatten()
for i, name in enumerate(names):
    cm = results[name]['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Malignant','Benign'],
                yticklabels=['Malignant','Benign'],
                linewidths=0.5, cbar=False, annot_kws={'size': 14})
    axes[i].set_title(
        f'{name}  (Accuracy = {results[name]["acc"]:.4f})',
        fontweight='bold', fontsize=11)
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
plt.suptitle('Confusion Matrices — All Four Models',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Precision / Recall / F1 Grouped Bar ───────────────────────────────
metrics_data = []
for name in names:
    cr = results[name]['cr']
    metrics_data.append({
        'Model':     name,
        'Precision': round(cr['weighted avg']['precision'], 4),
        'Recall':    round(cr['weighted avg']['recall'],    4),
        'F1-Score':  round(cr['weighted avg']['f1-score'],  4)
    })

mdf = pd.DataFrame(metrics_data).set_index('Model')
print(mdf.to_string())

ax = mdf.plot(kind='bar', figsize=(9, 5),
              color=['#4C72B0','#55A868','#C44E52'],
              edgecolor='white', width=0.65)
plt.xticks(rotation=0)
plt.ylim(0.88, 1.01)
plt.ylabel('Score', fontsize=12)
plt.title('Weighted Precision / Recall / F1-Score by Model',
          fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Summary Table & Best Model Justification

In [ ]:
summary = []
for name in names:
    cr = results[name]['cr']
    summary.append({
        'Model':             name,
        'Accuracy':          f"{results[name]['acc']*100:.2f}%",
        'Precision (wtd)':   f"{cr['weighted avg']['precision']:.4f}",
        'Recall (wtd)':      f"{cr['weighted avg']['recall']:.4f}",
        'F1-Score (wtd)':    f"{cr['weighted avg']['f1-score']:.4f}",
    })

sdf = pd.DataFrame(summary).set_index('Model')
print('\n===== FINAL COMPARISON TABLE =====')
print(sdf.to_string())
print(f'\n🏆 Best Model: ANN  (Accuracy 96.49%, F1 0.9647)')

## 7. Conclusion

| Model | Accuracy | F1 (weighted) |
|---|---|---|
| Decision Tree | 91.23% | 0.9118 |
| Naive Bayes   | 93.86% | 0.9383 |
| KNN           | 95.61% | 0.9558 |
| **ANN**       | **96.49%** | **0.9647** |

### Why ANN Performed Best

1. **Non-linear decision boundaries** — The two hidden layers (64 → 32 neurons, ReLU) learn complex, non-linear feature interactions that simpler models cannot capture.
2. **High-dimensional data** — With 30 continuous features the MLP can leverage all dimensions simultaneously via backpropagation; KNN and Decision Tree suffer more from the curse of dimensionality.
3. **Feature scaling** — StandardScaler was applied before ANN training, ensuring all gradients operate on the same scale and the Adam optimiser converges quickly.
4. **Recall for Malignant class = 0.98** — In a medical diagnosis context this is crucial: the model misses only 1 malignant case out of 42, minimising dangerous false negatives.